# Predict low-rating risk for movies on Snowflake

Which movies with an average rating above 3 in the previous 60 days will receive any rating below 3 in the next 60 days? Train Kumo Engine on synthetic movies, users and ratings, then score newly added movies.

Use a GPU Snowflake notebook with supported local SSD scratch and the [Snowflake integration](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/snowflake-notebook). Metrics on these synthetic data are demonstration only.


## Install

Install the Snowflake integration, then restart the kernel before continuing with configuration.

In [ ]:
%pip install "kumo-engine[snowflake]"

## Set the example parameters

Choose your own warehouse and schema, the future scoring anchor, and local paths. The 0.5 threshold converts low-rating probabilities to decisions; it does not change their ranking.


In [ ]:
from datetime import datetime
from pathlib import Path
from uuid import uuid4

WAREHOUSE_NAME = "<YOUR_WAREHOUSE>"
DATABASE_NAME = "<YOUR_DATABASE>"
SCHEMA_NAME = "<YOUR_SCHEMA>"  # Use a schema containing only these generated tables.
PREDICTION_TIME = datetime(2006, 1, 5)  # Follows the initial generated history.
BINARY_THRESHOLD = 0.5
OUTPUT_PARENT = Path.cwd() / "movie_risk_runs"
SCRATCH_PARENT = Path("/tmp")  # Confirm this is supported local SSD storage.
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
run_dir = OUTPUT_PARENT / uuid4().hex
run_dir.mkdir(exist_ok=False)


## Generate synthetic example data

Run the next cell to create initial history and reserve a new movie cohort with observed ratings. You can skim generation; the graph workflow starts below. Load only the initial files before fitting. The same noisy rating process generates both cohorts.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
users_data = pd.DataFrame({"userID": np.arange(240), "preferred_genre": np.arange(240) % 5})

def make_movies_and_ratings(first_movie, movie_count, rating_count, start, days):
    movie_ids = np.arange(first_movie, first_movie + movie_count)
    quality = rng.uniform(2, 5, movie_count)
    movies_data = pd.DataFrame({"MovieID": movie_ids, "genre": movie_ids % 5,
                                "quality": quality, "YearOfRelease": 1995})
    user_ids = rng.integers(0, len(users_data), rating_count)
    movie_position = rng.integers(0, movie_count, rating_count)
    selected_movies = movie_ids[movie_position]
    values = quality[movie_position] + (user_ids % 5 == selected_movies % 5)
    ratings_data = pd.DataFrame({
        "userID": user_ids, "movieID": selected_movies,
        "date": pd.Timestamp(start) + pd.to_timedelta(rng.integers(0, days, rating_count), unit="D"),
        "rating": np.clip(values + rng.normal(0, 0.8, rating_count), 1, 5).round(1),
    })
    return movies_data, ratings_data

movies_data, ratings_data = make_movies_and_ratings(0, 100, 24000, "2000-01-01", 6 * 365)
new_movies, new_ratings = make_movies_and_ratings(
    100, 12, 2400, pd.Timestamp(PREDICTION_TIME) - pd.Timedelta(days=59), 59)
assert set(new_movies.MovieID).isdisjoint(movies_data.MovieID)
assert new_ratings.movieID.isin(new_movies.MovieID).all()
assert new_ratings.userID.isin(users_data.userID).all()
assert ratings_data.date.lt(pd.Timestamp(PREDICTION_TIME)).all()
assert new_ratings.date.lt(pd.Timestamp(PREDICTION_TIME)).all()
recent_mean = new_ratings.groupby("movieID").rating.mean()
new_movie_ids = recent_mean[recent_mean > 3].index.to_numpy()
assert len(new_movie_ids) >= 2
frames = {"Users": users_data, "Movies": movies_data, "Ratings": ratings_data}
reserved = {"Movies": new_movies, "Ratings": new_ratings}
initial_dir = run_dir / "initial"
initial_dir.mkdir()
for name, frame in frames.items():
    frame.to_parquet(initial_dir / f"{name}.parquet", index=False)
print("Load only these initial files before training:", initial_dir)
print({name: len(frame) for name, frame in frames.items()})
print("Reserved eligible new movies:", len(new_movie_ids))


## Connect to your loaded tables

Manually load each `initial/` Parquet file into the configured schema using its filename stem. Preserve numeric key types and date values; quote the mixed-case table/column names in Snowflake. The generator performs no upload. The runtime OAuth identity needs read access to all three tables.


In [ ]:
import tempfile
import kumo.engine as kumo
from snowflake.snowpark.context import get_active_session

session = get_active_session()
scratch = Path(tempfile.mkdtemp(prefix="movie-risk-kumo-", dir=SCRATCH_PARENT))
kumo.configure(scratch_dir=scratch)
connector = kumo.SnowflakeConnector.from_spcs_env(
    warehouse=WAREHOUSE_NAME, database=DATABASE_NAME, schema_name=SCHEMA_NAME,
)

## Inspect the source

Check the columns needed by the graph before materialization.

In [ ]:
print('Tables:', connector.table_names())

for table_name in ('Movies', 'Users', 'Ratings'):
    for col in connector[table_name].columns:
        print(f'{table_name}.{col.name}', col.dtype,
              col.stype or '- (needs column statistics)')

## Define the tables

Use movie and user IDs as keys, and the rating date as the time when a rating becomes available.
The graph omits movie titles to keep this example focused on numerical and categorical features.

In [ ]:
from kumoapi.typing import Dtype, Stype

movies = kumo.Table.from_source_table(
    connector["Movies"], ["MovieID", "YearOfRelease"], primary_key="MovieID")
users = kumo.Table.from_source_table(connector["Users"], primary_key="userID")
# Foreign keys connect the ratings in Build the graph below.
# Declare date explicitly; this also permits date-compatible strings after loading.
ratings = (kumo.Table.from_source_table(connector["Ratings"], time_column="date")
           .infer_metadata()
           .with_column("movieID", stype=Stype.ID)
           .with_column("userID", stype=Stype.ID)
           .with_column("date", dtype=Dtype.date))
for label, table in [("movies", movies), ("users", users), ("ratings", ratings)]:
    print(label, [(column.name, column.stype) for column in table.definition.cols])

## Build the graph

Validate the two relationships from ratings to movies and users.

In [ ]:

from kumo.engine import Edge

graph = kumo.Graph(
    tables=[movies, users, ratings],
    edges=[
        Edge(src_table='Ratings', fkey='movieID', dst_table='Movies'),
        Edge(src_table='Ratings', fkey='userID', dst_table='Users'),
    ],
)
graph.validate()
print('graph valid. snapshot_id:', graph.id)

## Inspect the graph

Inspect the actual graph’s tables and relationships before materialization. The [graph visualization](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs) call is a commented preview because it is unavailable in the supported API.

In [ ]:
# graph.visualize(show_cols=True)

## Define the prediction question

Select movies whose average rating during the previous 60 days is strictly above 3. Predict whether at least one rating strictly below 3 arrives during the following 60 days. This is movie-level risk, not a recommendation for an individual user. See [predictive queries](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/predictive-queries).

In [ ]:
QUERY = """
PREDICT COUNT_DISTINCT(Ratings.rating WHERE Ratings.rating < 3, 0, 60, days) > 0
FOR EACH Movies.MovieID
WHERE AVG(Ratings.rating, -60, 0, days) > 3
""".strip()
print(QUERY)

pquery = kumo.PredictiveQuery(graph=graph, query=QUERY)
pquery.validate()
print('\ntask type:', pquery.get_task_type())

## Prepare historical training rows

Use approximate chronological 70/15/15 splits so validation and test follow training. Anchors are 60 days apart over the latest 720 days, and the plan leaves each target window observable. Inspect the actual split dates and populations (`0` train, `1` validation, `2` test).

In [ ]:
from kumoapi.model_plan import TrainingTableGenerationPlan
from kumo.engine.trainer.output_config import TrainingTableOutputConfig

snapshot = graph.snapshot()
generation_plan = TrainingTableGenerationPlan(
    split="ApproxDateOffsetSplit([0.7, 0.15, 0.15])",
    train_start_offset=720, train_end_offset=0, timeframe_step=60,
)
train_table = pquery.generate_training_table(snapshot, plan=generation_plan)
local_output = kumo.FileConnector(root_dir=str(run_dir))
train_table.export(TrainingTableOutputConfig(connector=local_output, table_name="training_rows/"))
training_rows = pd.read_parquet(run_dir / "training_rows")
print(training_rows.groupby("SPLIT").agg(rows=("ENTITY1", "size"), first_anchor=("TIMESTAMP", "min"), last_anchor=("TIMESTAMP", "max")))

## Configure the model

Use the suggested FAST plan, keep held-out splits separate from fitting, and inspect the selected validation metric.

In [ ]:
from kumoapi.model_plan import RunMode

plan = pquery.suggest_model_plan(snapshot, run_mode=RunMode.FAST)
plan.training_job.refit_trainval = False
plan.training_job.refit_full = False
trainer = kumo.Trainer(plan)
print("Validation selection metric:", plan.training_job.tune_metric)

## Train and retain the model

Fit the classifier and export its complete bundle to this run’s local artifact directory.


In [ ]:
result = trainer.fit(train_table, snapshot=snapshot,
    export_config=kumo.ModelOutputConfig(connector=local_output, path="model"))
print("Retained local model:", run_dir / "model")


## Evaluate

**Synthetic metrics — demonstration only.** AUROC measures how well the model ranks movies that later receive a low rating above those that do not; average precision summarizes precision across recall levels. Use the validation stage for model choices and reserve test results for final assessment. These metrics do not establish quality on another catalog.

In [ ]:
print("Synthetic metrics — demonstration only:", result.metrics)


## Batch predictions

Append the reserved movies and their observed ratings after fitting. Existing users are reused, and every new rating precedes the scoring anchor. Score only new movies whose observed 60-day average exceeds 3; no future ratings enter the scoring graph.

In [ ]:
append_dir = run_dir / "append"
append_dir.mkdir(exist_ok=False)
for name, frame in reserved.items():
    frame.to_parquet(append_dir / f"{name}.parquet", index=False)
print("Append these files once to the matching source tables:", append_dir)


### Load and verify the appended rows

Manually append the two `append/` files to the matching existing `Movies` and `Ratings` tables. Append once, preserve existing rows, and do not reload the users table. Continue after loading; the checks below compare the full small tables with the expected original-plus-new data.


In [ ]:
# Mixed-case Snowflake table names are quoted explicitly.
for name, initial in frames.items():
    identifiers = ['"' + value.replace('"', '""') + '"'
                   for value in (DATABASE_NAME, SCHEMA_NAME, name)]
    loaded = session.table(identifiers).to_pandas()
    expected = pd.concat([initial, reserved[name]], ignore_index=True) if name in reserved else initial
    assert len(loaded) == len(expected), f"Complete the one-time append for {name}."
    if name == "Ratings":
        loaded["date"] = pd.to_datetime(loaded["date"])
    ordering = list(expected.columns)
    pd.testing.assert_frame_equal(
        loaded[ordering].sort_values(ordering).reset_index(drop=True),
        expected.sort_values(ordering).reset_index(drop=True),
        check_dtype=False, check_exact=False,
    )


### Prepare the new-movie population

Refresh the source connector and snapshot with unchanged feature definitions. Use only eligible new movie IDs and their anchor in the prediction table.


In [ ]:
from dataclasses import replace

scoring_connector = kumo.SnowflakeConnector.from_spcs_env(
    warehouse=WAREHOUSE_NAME, database=DATABASE_NAME, schema_name=SCHEMA_NAME)
scoring_graph = kumo.Graph(
    tables=[replace(table, connector=scoring_connector) for table in graph.tables],
    edges=graph.edges,
)
scoring_graph.validate()
scoring_snapshot = scoring_graph.snapshot()
assert scoring_snapshot.id != snapshot.id
scoring_query = kumo.PredictiveQuery(scoring_graph, QUERY)
scoring_query.validate()
scoring_population = pd.DataFrame({"ENTITY1": new_movie_ids, "TIMESTAMP": PREDICTION_TIME})
assert len(scoring_population) >= 2 and scoring_population.ENTITY1.is_unique
assert not any(column.startswith("TARGET") for column in scoring_population)
scoring_population.to_parquet(run_dir / "scoring_population.parquet", index=False)
prediction_table = kumo.PredictionTable.from_source_table(
    local_output["scoring_population"], pq=scoring_query, snapshot=scoring_snapshot,
    plan=trainer.training_table_generation_plan,
)
population = pd.read_parquet(prediction_table.table_path)
assert len(population) == len(scoring_population)
assert set(population.ENTITY1) == set(new_movie_ids)
print(population.head(10).to_string(index=False))


### Score the new movies

Run the fitted classifier on the updated graph without retraining.


In [ ]:
pred_result = trainer.predict(
    prediction_table, snapshot=scoring_snapshot,
    binary_classification_threshold=BINARY_THRESHOLD,
)


### Inspect and save results

Expect one row per eligible new movie. `True_PROB` estimates the chance of any rating below 3 within 60 days; `TARGET_PRED` is true strictly above the configured threshold. Verify IDs and save the complete batch.


In [ ]:
scores = pred_result.frame
assert len(scores) == len(population) and scores.ENTITY.is_unique
assert set(scores.ENTITY) == set(population.ENTITY1)
assert scores.True_PROB.between(0, 1).all()
assert scores.TARGET_PRED.astype(bool).eq(scores.True_PROB > BINARY_THRESHOLD).all()
print(scores.head(10).to_string(index=False))
output_path = run_dir / "movie_risk_predictions.parquet"
scores.to_parquet(output_path, index=False)
print("Saved:", output_path)
top = scores.sort_values("True_PROB", ascending=False).head(5)
print("New movies most likely to receive a rating below 3 in the next 60 days:")
print(top[["ENTITY", "True_PROB"]].to_string(index=False))


## Retain and reuse the model

Copy the complete `run_dir / "model"` directory and prediction file to durable storage before the notebook stops. In a later session, configure scratch, reconnect and rebuild the compatible graph/snapshot, then load the trusted bundle with `kumo.Trainer.load()`. The bundle does not include source data. See [model reuse](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/publish-and-retrieve-models).